In [2]:
import pandas as pd

from unibench.benchmarks_zoo.registry import list_benchmarks
from unibench.models_zoo.registry import list_models
from unibench.output import OutputHandler
import seaborn as sns
import matplotlib.pyplot as plt

models = [
    'qwen_3_2b',
    # 'qwen_3_30b_a3b',
    'qwen_3_32b',
    'qwen_3_8b', 
    'qwen_3_4b', 
    'siglip2_so400_14_378', 
    'siglip_so400_14', 
    'clip_vitL14',
    'eva01_vitG14_plus_2b', 
    'llava_1_5_13b', 
    'llava_1_5_7b', 
    'llava_1_6_mistral_7b',
    'llava_1_6_vicuna_13b', 
    'llava_1_6_vicuna_7b', 
    'llava_next_llama_8b', 
    'aya_vision_32b',
    'aya_vision_8b',
    'gemma3_4b',
    'gemma3_12b',
    'paligemma_3b_mix_224', 
    'paligemma2_3b_mix_224', 
    'paligemma2_10b_mix_224', 
    'paligemma2_10b_mix_448', 
    'paligemma2_28b_mix_224', 
    'paligemma2_3b_mix_448', 
    'paligemma_3b_mix_448'
]


outputhandler = OutputHandler(output_dir='/storage/home/hcoda1/6/haltahan6/scratch/unibench/tests/haider/ga_cluster/script_outputs', download_all_precomputed=False)

outputhandler.load_all_csvs(
    model_names=models,
)

results = outputhandler.query(**{"model_name": models})
from unibench.common_utils.utils import get_model_mappings
model_mappings = get_model_mappings('model_type')
results['model_type'] = results["model_name"].map(model_mappings)
model_mappings = get_model_mappings('name')
results['model_name'] = results["model_name"].map(model_mappings)

from unibench.common_utils.utils import get_benchmark_mappings

# Assign num_classes based on benchmark_name
results.loc[~results['benchmark_name'].isin(['countbench', 'vg_relation', 'flickr30k_order', 'sugarcrepe', 'bivlc', 'winoground', 'vg_attribution', 'coco_order']), 'num_classes'] = results.loc[~results['benchmark_name'].isin(['countbench', 'vg_relation', 'flickr30k_order', 'sugarcrepe', 'bivlc', 'winoground', 'vg_attribution', 'coco_order']), 'benchmark_name'].str.split('_').str[-1].astype(int)
results.loc[~results['benchmark_name'].isin(['countbench', 'vg_relation', 'flickr30k_order', 'sugarcrepe', 'bivlc', 'winoground', 'vg_attribution', 'coco_order']), 'benchmark_name'] = results.loc[~results['benchmark_name'].isin(['countbench', 'vg_relation', 'flickr30k_order', 'sugarcrepe', 'bivlc', 'winoground', 'vg_attribution', 'coco_order']), 'benchmark_name'].str.rsplit('_', n=1).str[0]
results.loc[results['benchmark_name'].isin(['countbench', 'vg_relation', 'flickr30k_order', 'sugarcrepe', 'bivlc', 'winoground', 'vg_attribution', 'coco_order']), 'num_classes'] = 0

benchmark_mappings = get_benchmark_mappings("capability")
results["capability"] = results["benchmark_name"].map(benchmark_mappings)
benchmark_mappings = get_benchmark_mappings("benchmark_type")
results["benchmark_type"] = results["benchmark_name"].map(benchmark_mappings)

In [ ]:
vllm_capability = ['relations', 'spatial understanding', 'pose detection', 'depth estimation', ]
contrastive_capability = ['corruption', 'imagenet', 'specifies classification', 'counting', 'geographic diversity', 'natural transformations', 'rendition', 'standard object recognition', 'challenging imagenet', 'satellite', 'texture detection', 'character recognition', 'medical', 'scene recognition']

from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Prepare the data
r = total_results[32]
cap_set = set(vllm_capability) | set(contrastive_capability)

classifier_data = r[
    (r["prompt"].notna()) &
    (r["capability"].notna()) &
    (r["capability"].isin(cap_set))
].copy()

# Binary target: 1 for vllm capabilities, 0 for contrastive capabilities
classifier_data["target"] = classifier_data["capability"].apply(
    lambda x: 1 if x in vllm_capability else 0
)

# Unique prompt-target pairs to avoid leakage
unique_prompts = classifier_data[["prompt", "target"]].drop_duplicates()

print(f"Total unique prompts: {len(unique_prompts)}")
print(f"VLLM capability prompts: {(unique_prompts['target'] == 1).sum()}")
print(f"Contrastive capability prompts: {(unique_prompts['target'] == 0).sum()}")

X = unique_prompts["prompt"].values
y = unique_prompts["target"].values

# TF-IDF + Logistic Regression in a single CV-safe pipeline
pipe = make_pipeline(
    TfidfVectorizer(max_features=8196, stop_words="english"),
    LogisticRegression(random_state=42, max_iter=2000),
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_validate(
    pipe,
    X,
    y,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1,
    return_train_score=False,
)

print("\n5-fold CV metrics (mean ± std):")
for m in ["accuracy", "precision", "recall", "f1"]:
    vals = scores[f"test_{m}"]
    print(f"  {m:>9}: {vals.mean():.4f} ± {vals.std():.4f}")

# Out-of-fold predictions for a full classification report
y_pred = cross_val_predict(pipe, X, y, cv=cv, n_jobs=-1)

print("\nOut-of-fold Classification Report:")
print(classification_report(y, y_pred, target_names=["Contrastive", "VLLM"]))